# Kafka Data Simulator

Sends simulated business events to Confluent Cloud Kafka topics using Spark.

In [1]:
# Confluent Cloud Configuration
KAFKA_BOOTSTRAP_SERVERS = "pkc-921jm.us-east-2.aws.confluent.cloud:9092"

KAFKA_API_KEY = "XDNCGI37HYWXKQ43"
KAFKA_API_SECRET = "cfltkC3TGvQcqDzS0S/se0r/QgHrzyxoZLXKS5JudeBGmnq9PmTpf+KzEYpZThdA"

# Topics
TOPIC_ORDERS_CREATED = "orders.created"
TOPIC_ORDERS_UPDATED = "orders.updated"
TOPIC_PAYMENTS = "payments.authorized"

# Simulation
NUM_EVENTS = 50

print(f"Bootstrap: {KAFKA_BOOTSTRAP_SERVERS}")
print(f"Topics: {TOPIC_ORDERS_CREATED}, {TOPIC_ORDERS_UPDATED}, {TOPIC_PAYMENTS}")

StatementMeta(, f3c8ca2e-3a8e-47f9-a8f4-59287a294ab0, 3, Finished, Available, Finished, False)

Bootstrap: pkc-921jm.us-east-2.aws.confluent.cloud:9092
Topics: orders.created, orders.updated, payments.authorized


In [2]:
# Kafka write options
kafka_options = {
    "kafka.bootstrap.servers": KAFKA_BOOTSTRAP_SERVERS,
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.sasl.jaas.config": f'org.apache.kafka.common.security.plain.PlainLoginModule required username="{KAFKA_API_KEY}" password="{KAFKA_API_SECRET}";'
}

print("Kafka options configured")

StatementMeta(, f3c8ca2e-3a8e-47f9-a8f4-59287a294ab0, 4, Finished, Available, Finished, False)

Kafka options configured


In [3]:
import json
import random
import uuid
from datetime import datetime, timezone
from pyspark.sql import Row

# Sample data
CUSTOMERS = [f"CUST-{i:04d}" for i in range(1, 20)]
PRODUCTS = [
    {"id": "PROD-001", "name": "Laptop Pro", "price": 1299.99},
    {"id": "PROD-002", "name": "Wireless Mouse", "price": 49.99},
    {"id": "PROD-003", "name": "USB-C Hub", "price": 79.99},
    {"id": "PROD-004", "name": "Monitor 27in", "price": 399.99},
    {"id": "PROD-005", "name": "Keyboard", "price": 149.99}
]
REGIONS = ["US-East", "US-West", "EU-West", "APAC"]
STATUSES = ["pending", "confirmed", "shipped", "delivered"]
PAYMENT_METHODS = ["credit_card", "paypal", "apple_pay"]

def now_iso():
    return datetime.now(timezone.utc).isoformat()

print("Sample data loaded")

StatementMeta(, f3c8ca2e-3a8e-47f9-a8f4-59287a294ab0, 5, Finished, Available, Finished, False)

Sample data loaded


In [4]:
# Generate events
print(f"Generating {NUM_EVENTS} events...")

orders_created = []
orders_updated = []
payments = []
existing_orders = []

for i in range(NUM_EVENTS):
    event_type = random.choices(["create", "update", "payment"], weights=[5, 3, 2])[0]
    
    if event_type == "create" or not existing_orders:
        order_id = f"ORD-{uuid.uuid4().hex[:8].upper()}"
        items = random.sample(PRODUCTS, k=random.randint(1, 3))
        total = sum(p["price"] * random.randint(1, 2) for p in items)
        
        event = {
            "event_type": "order.created",
            "event_time": now_iso(),
            "order_id": order_id,
            "customer_id": random.choice(CUSTOMERS),
            "region": random.choice(REGIONS),
            "items": [{"product_id": p["id"], "name": p["name"], "price": p["price"]} for p in items],
            "total_amount": round(total, 2)
        }
        orders_created.append(Row(key=order_id, value=json.dumps(event)))
        existing_orders.append({"id": order_id, "amount": total})
        
    elif event_type == "update":
        order = random.choice(existing_orders)
        event = {
            "event_type": "order.updated",
            "event_time": now_iso(),
            "order_id": order["id"],
            "new_status": random.choice(STATUSES)
        }
        orders_updated.append(Row(key=order["id"], value=json.dumps(event)))
        
    else:
        order = random.choice(existing_orders)
        event = {
            "event_type": "payment.authorized",
            "event_time": now_iso(),
            "payment_id": f"PAY-{uuid.uuid4().hex[:8].upper()}",
            "order_id": order["id"],
            "amount": order["amount"],
            "method": random.choice(PAYMENT_METHODS)
        }
        payments.append(Row(key=order["id"], value=json.dumps(event)))

print(f"Generated: {len(orders_created)} orders, {len(orders_updated)} updates, {len(payments)} payments")

StatementMeta(, f3c8ca2e-3a8e-47f9-a8f4-59287a294ab0, 6, Finished, Available, Finished, False)

Generating 50 events...
Generated: 15 orders, 26 updates, 9 payments


In [5]:
# Send to Kafka using Spark
def send_to_topic(rows, topic):
    if not rows:
        print(f"{topic}: 0 events (skipped)")
        return
    df = spark.createDataFrame(rows)
    df.write.format("kafka").options(**kafka_options).option("topic", topic).save()
    print(f"{topic}: {len(rows)} events sent")

print("Sending to Kafka...")
send_to_topic(orders_created, TOPIC_ORDERS_CREATED)
send_to_topic(orders_updated, TOPIC_ORDERS_UPDATED)
send_to_topic(payments, TOPIC_PAYMENTS)

print("")
print("=== Done ===")

StatementMeta(, f3c8ca2e-3a8e-47f9-a8f4-59287a294ab0, 7, Finished, Available, Finished, False)

Sending to Kafka...
orders.created: 15 events sent
orders.updated: 26 events sent
payments.authorized: 9 events sent

=== Done ===
